In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score, roc_curve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

df = pd.read_excel(r"C:\Users\LENOVO\Desktop\Customer Churn Analysis Project\Dataset\Telco_customer_churn.xlsx")

df.drop("CustomerID", axis=1, inplace=True)

drop_cols = [
    'Count',
    'Country',
    'State',
    'Zip Code',
    'Lat Long',
    'Churn Reason',
    'Churn Value',
    'Churn Score',
    'CLTV'
]

df.drop(columns=drop_cols, inplace=True)

df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")

df["Total Charges"] = df["Total Charges"].fillna(df["Total Charges"].median())

#ENCODING 

binary_cols = ['Partner', 'Dependents', 'Phone Service', 'Paperless Billing', 'Churn Label']

for col in binary_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

categorical_cols = [
    'Internet Service',
    'Contract',
    'Payment Method'
]

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

categorical_cols = df.select_dtypes(include=['object','string']).columns

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# ML TRAINING
X = df.drop("Churn Label", axis=1)

y = df["Churn Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# LOGISTIC REGRESSION
lr = LogisticRegression(max_iter=5000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

# RANDOM FOREST
rf = RandomForestClassifier()

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

# XGBoost Classifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

# SVM

svm = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

# MODEL EVALUATION

#ACCURACY SCORE 

metrics_df = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "SVM"
    ],
    "Accuracy": [
        accuracy_score(y_test,y_pred_lr),
        accuracy_score(y_test,y_pred_rf),
        accuracy_score(y_test,y_pred_xgb),
        accuracy_score(y_test,y_pred_svm)
    ],
    "Precision": [
        precision_score(y_test,y_pred_lr),
        precision_score(y_test,y_pred_rf),
        precision_score(y_test,y_pred_xgb),
        precision_score(y_test,y_pred_svm)
    ],
    "Recall": [
        recall_score(y_test,y_pred_lr),
        recall_score(y_test,y_pred_rf),
        recall_score(y_test,y_pred_xgb),
        recall_score(y_test,y_pred_svm)
    ],
    "F1 Score": [
        f1_score(y_test,y_pred_lr),
        f1_score(y_test,y_pred_rf),
        f1_score(y_test,y_pred_xgb),
        f1_score(y_test,y_pred_svm)
    ]
})

# predicted churn distribution 

distribution_list = []

models = {
    "Logistic Regression": y_pred_lr,
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb,
    "SVM": y_pred_svm
}

for model_name,preds in models.items():

    churn = np.sum(preds==1)
    stay = np.sum(preds==0)

    distribution_list.append(
        [model_name,"Likely to Churn",churn]
    )

    distribution_list.append(
        [model_name,"Likely to Stay",stay]
    )

distribution_df = pd.DataFrame(
    distribution_list,
    columns=["Model","Prediction","Count"]
)


#CONFUSION MATRIX

cm_data = []

for model_name,preds in models.items():

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        preds
    ).ravel()

    cm_data.extend([
        [model_name,"True Negative",tn],
        [model_name,"False Positive",fp],
        [model_name,"False Negative",fn],
        [model_name,"True Positive",tp]
    ])

confusion_df = pd.DataFrame(
    cm_data,
    columns=[
        "Model",
        "Confusion_Type",
        "Count"
    ]
)

# Actual vs Predicted Churn

actual_pred_list = []

for model_name,preds in models.items():

    actual_churn = np.sum(y_test==1)
    predicted_churn = np.sum(preds==1)

    actual_pred_list.append([
        model_name,
        "Actual Churn",
        actual_churn
    ])

    actual_pred_list.append([
        model_name,
        "Predicted Churn",
        predicted_churn
    ])

actual_pred_df = pd.DataFrame(
    actual_pred_list,
    columns=[
        "Model",
        "Category",
        "Count"
    ]
)

# Feature Importance

#RANDOM FOREST 
rf_importance = pd.DataFrame({
    "Feature":X.columns,
    "Importance":rf.feature_importances_,
    "Model":"Random Forest"
})

#XG BOOST 
xgb_importance = pd.DataFrame({
    "Feature":X.columns,
    "Importance":xgb.feature_importances_,
    "Model":"XGBoost"
})

#Logistic Regression
lr_importance = pd.DataFrame({
    "Feature":X.columns,
    "Importance":np.abs(
        lr.coef_[0]
    ),
    "Model":"Logistic Regression"
})

#SVM
svm_importance = pd.DataFrame({
    "Feature":["Not Available"],
    "Importance":[0],
    "Model":["SVM"]
})

importance_df = pd.concat([
    rf_importance,
    xgb_importance,
    lr_importance,
    svm_importance
])

importance_df = (
    importance_df
    .sort_values(
        ["Model","Importance"],
        ascending=[True,False]
    )
    .groupby("Model")
    .head(5)
)

with pd.ExcelWriter(
    "ABC123.xlsx",
    engine="openpyxl"
) as writer:

    metrics_df.to_excel(
        writer,
        sheet_name="Model Metrics",
        index=False
    )

    distribution_df.to_excel(
        writer,
        sheet_name="Prediction Distribution",
        index=False
    )

    confusion_df.to_excel(
        writer,
        sheet_name="Confusion Matrix",
        index=False
    )

    actual_pred_df.to_excel(
        writer,
        sheet_name="Actual vs Predicted",
        index=False
    )

    importance_df.to_excel(
        writer,
        sheet_name="Feature Importance",
        index=False
    )


In [ ]:
pip install openpyxl